# GSE249479 | Human HSPC inflammatory conditions

**Public application gallery — ScGeo 0.2.0 or opt-in 0.2.1.** Reuse the exact revision representation and expression objects for 34,432 retained cells. This is a new presentation of the existing analysis, not a rerun of the manuscript. See the [application guide](README.md) before executing. The original study's sample labels and souporcell assignments do not establish independent biological replication; PBS/TNF/LPS comparisons remain **descriptive only**.

No RNA velocity is available and no cell-to-cell temporal transitions are inferred. The compact expression file is opened in backed read-only mode to verify cell identities; the geometry-only H5AD contains no genes or loadings. Exact gene attribution is not performed.

In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_applications.py").is_file()), None)
if root is None:
    raise RuntimeError("Open from the companion repository checkout.")
sys.path.insert(0, str(root / "scripts"))
from scgeo_v020_applications import Application, PRIMARY, OTHER, install_graph, plot_evidence, build_pca50_graph
from scgeo_v020_applications import plot_profile, apply_profile
app = Application("B")
adata, annotation = app.load()
print(adata.shape, list(adata.obsm))

## 1. Common display coordinates and published annotation mapping

In [ ]:
require_labels = adata.obs["marker_inferred_label"].astype(str).map(app.config["detailed_state_mapping"])
assert require_labels.notna().all()
adata.obs["application_state"] = pd.Categorical(require_labels,
    categories=app.config["detailed_state_order"])
adata.obs["application_coarse"] = adata.obs["application_state"].astype(str).map(
    app.config["coarse_state_mapping"])
assert adata.obs["application_coarse"].notna().all()
assert set(adata.obs["condition"].astype(str)) == {"PBS", "TNF", "LPS"}
assert set(adata.obs["inference_status"].astype(str)) == {"descriptive_only"}
groups = ["PBS", "TNF", "LPS"]
basis = "umap_pca50"
counts = pd.crosstab(adata.obs["condition"], adata.obs["application_state"])
display(counts)
counts.to_csv(app.out / "condition_state_counts.csv")
app.save(sg.pl.embedding_facets(adata, basis=basis, facet="condition",
    facet_order=groups, color="application_state", shared_limits=True, col_wrap=3,
    max_points=12000, point_size=3, random_state=1729,
    title="GSE249479 | Inflammatory conditions (descriptive)",
    return_result=True, show=False), "condition_facets", "Fixed frozen UMAP for display")

## 2. Inspect previously computed molecular scores

These are the original per-cell signature scores, not newly inferred drivers or causal effects. Every facet of a given score shares color scaling.

In [ ]:
for score in ["score_tnf_nfkb_response", "score_ap1_stress_response",
              "score_interferon_response"]:
    app.save(sg.pl.embedding_facets(adata, basis=basis, facet="condition",
        facet_order=groups, color=score, shared_limits=True, col_wrap=3,
        max_points=10000, point_size=3, random_state=1729, cmap="viridis",
        title=f"GSE249479 | {score.removeprefix('score_')} (stored score)",
        return_result=True, show=False), score, "Precomputed descriptive signature; not gene attribution")

## 3. Relative embedding density and annotation associations

Per-group density is normalized by each group's own maximum: compare spatial shape, not abundance. The alluvial links connect alternative labels for the same cells, not developmental trajectories.

In [ ]:
app.save(sg.pl.embedding_density(adata, "condition", basis=basis, groups=groups,
    normalize="per_group", gridsize=80, contour=True, smooth_k=3, log1p=True,
    return_result=True, show=False), "relative_density", "Descriptive density shape, not absolute abundance")
app.save(sg.pl.state_flow_alluvial(adata, columns=["condition", "application_state",
    "application_coarse"], color_by="source", min_count=10, normalize=True,
    title="GSE249479 | Condition and annotation associations",
    return_result=True, show=False), "annotation_associations",
    "Same-cell associations; captured cells are not independent biological samples")

## 4. Build a new descriptive graph, without refitting PCA or scVI

The saved HSPC representation lacks a cell graph. Build a 30-neighbor graph on the existing PCA50 coordinates with Scanpy in a temporary coordinate-only AnnData; this is a new graph calculation for the gallery. The frozen object retains diffusion coordinates but not diffusion eigenvalues. Isolating the PCA50 input avoids Scanpy's unrelated diffusion metadata initialization without modifying the source. Aggregate its symmetric nonnegative edge weights between states and normalize by the geometric mean of total incident weights. Use the PAGA-style plotting interface on this **descriptive state adjacency**, not the Scanpy PAGA estimator. No directed edges or lineage interpretation.

In [ ]:
graph = build_pca50_graph(adata, n_neighbors=30, random_state=1729)
names = install_graph(app, adata, graph,
    "pooled", "New Scanpy neighbors (30, PCA50, euclidean, seed 1729), all PBS/TNF/LPS cells; isolated scratch object")
display(pd.DataFrame(adata.uns["application_adjacency"]["connectivities"]))
del graph

## 5. Redraw the frozen TNF and LPS evidence

Each original detailed-state evidence table is checked against the exact cell counts and source scope. Its SHA256 is recorded (observed source hash, not independently certified against a separate archive). No cell-bootstrap interval is presented as biological uncertainty. The response compass reads PCA50 effect estimates from the table; its arrows connect observed display centers, not quantitative PCA displacements. Velocity alignment and OOD evidence are unavailable.

In [ ]:
inputs = [(treated, app.b_evidence(adata, treated)) for treated in ["TNF", "LPS"]]
profile = plot_profile(adata, [frame for _, frame in inputs],
                       app.config["detailed_state_order"], "condition")
for treated, frame in inputs:
    report = plot_evidence(app, adata, frame, "PBS", treated, "condition",
                           basis, f"PBS_vs_{treated}", names, profile=profile)
    display(report[["state", "n_cells0", "n_cells1", "normalized_delta_norm", "effect_status"]])
    subset = adata[adata.obs["condition"].isin(["PBS", treated])].copy()
    result = sg.pl.geometry_dynamics_atlas(subset, basis=basis, color="application_state",
        node_key="application_state", condition_key="condition", group0="PBS", group1=treated,
        panels=("states", "evidence"), report=report,
        layout="adaptive" if profile else "grid", effect_metric="normalized_delta_norm",
        title=f"PBS vs {treated} | Geometry-only evidence atlas",
        max_points=8000, random_state=1729, show=False)
    app.save(apply_profile(result, profile, effect_axis="evidence"), f"PBS_vs_{treated}_atlas",
        "Observed states and frozen descriptive evidence; no velocity or graph panel")
    del subset

## 6. Provenance and limits

The exact expression-to-PCA/SVD transformation is not available in these H5ADs. Recover its feature order, input normalization and weights before attempting signed attribution. No pseudo-replicates or velocity estimates are introduced. Keep the original comparator and inference limits in the [frozen workflow](../../public_validation/gse249479/README.md) (navigate from the repository root if using a notebook viewer).

In [ ]:
manifest = app.finish()
display({k: manifest[k] for k in ["status", "dataset", "plot_manifests", "figures", "source_tables"]})